In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path("/data/raw/fts_flows_raw.csv")
OUT_CLEAN = Path("/data/interim/fts_flows_clean.csv")
OUT_CM    = Path("/data/interim/fts_flows_country_month.csv")

TIER1    = {"BHS","BLZ","BRB","CRI","CUB","DOM","GTM","GUY",
            "HND","HTI","JAM","NIC","PAN","PRI","SLV","SUR","TTO"}
PRESSURE = {"COL","VEN"}
COUNTRY_ROLE = {c: "tier1" for c in TIER1}
COUNTRY_ROLE.update({c: "pressure" for c in PRESSURE})

# ── Load & filter ─────────────────────────────────────────────────────────────
raw = pd.read_csv(RAW)
paid = raw[
    (raw["boundary"] == "incoming") &
    (raw["funding_status"] == "paid") &
    (raw["dest_usage_year"].between(2019, 2024)) &
    (raw["amount_usd"] > 0)
].copy()
paid = paid[paid["iso3"].isin(TIER1 | PRESSURE)].copy()

# ── Dates ─────────────────────────────────────────────────────────────────────
paid["flow_date"] = pd.to_datetime(paid["flow_date"], errors="coerce")
paid["month"] = paid["flow_date"].dt.month
mismatch = paid["flow_date"].dt.year != paid["dest_usage_year"]
paid.loc[mismatch, "month"] = 6
paid["year"] = paid["dest_usage_year"].astype(int)
paid["country_role"] = paid["iso3"].map(COUNTRY_ROLE)

# ── Flags ─────────────────────────────────────────────────────────────────────
paid["is_cerf"]    = paid["source_org"].str.contains("Central Emergency", na=False)
paid["is_echo"]    = paid["source_org"].str.contains("Humanitarian Aid and Civil Protection", na=False)
paid["is_private"] = paid["source_org"].str.contains("Private|individual", case=False, na=False)

# ── source_country: fill nulls ────────────────────────────────────────────────
def fill_source_country(row):
    if pd.notna(row["source_country"]):
        return row["source_country"]
    return "Private" if row["is_private"] else "Multilateral"

paid["source_country"] = paid.apply(fill_source_country, axis=1)
paid["sector"] = paid["sector"].fillna("Unearmarked")

# ── Step 1: Deduplicate exact rows ────────────────────────────────────────────
DUP_COLS = ["iso3","year","month","source_org","source_country",
            "dest_org","sector","funding_status","boundary"]

deduped = (paid
    .groupby(DUP_COLS, dropna=False)
    .agg(
        amount_usd   = ("amount_usd", "sum"),
        is_cerf      = ("is_cerf", "max"),
        is_echo      = ("is_echo", "max"),
        country_role = ("country_role", "first"),
    )
    .reset_index())

print(f"Rows before dedup: {len(paid)}")
print(f"Rows after dedup:  {len(deduped)}  ({len(paid)-len(deduped)} exact duplicates removed)")

# ── File 1: fts_flows_clean.csv ───────────────────────────────────────────────
clean_cols = ["iso3","year","month","source_country","source_org",
              "dest_org","sector","amount_usd","is_cerf","is_echo","country_role"]

clean = (deduped[clean_cols]
         .sort_values(["iso3","year","month","amount_usd"], ascending=[True,True,True,False])
         .reset_index(drop=True))

clean.to_csv(OUT_CLEAN, index=False)
print(f"\nfts_flows_clean.csv: {clean.shape}")
print(f"source_country nulls remaining: {clean['source_country'].isna().sum()}")
print(f"\nTop source_country values:\n{clean['source_country'].value_counts().head(12).to_string()}")

# ── File 2: fts_flows_country_month.csv ──────────────────────────────────────
def sector_mix(s):
    vals = sorted(s.dropna().unique())
    return " | ".join(vals) if vals else np.nan

cm = (deduped
    .groupby(["iso3","year","month","country_role"])
    .agg(
        funding_usd = ("amount_usd", "sum"),
        flow_count  = ("amount_usd", "count"),
        has_cerf    = ("is_cerf", "max"),
        has_echo    = ("is_echo", "max"),
        sector_mix  = ("sector", sector_mix),
    )
    .reset_index()
    .sort_values(["iso3","year","month"])
    .reset_index(drop=True))

cm["has_cerf"] = cm["has_cerf"].astype(int)
cm["has_echo"] = cm["has_echo"].astype(int)

cm.to_csv(OUT_CM, index=False)
print(f"\nfts_flows_country_month.csv: {cm.shape}")

# ── Validation ────────────────────────────────────────────────────────────────
t1_clean = clean[clean["country_role"] == "tier1"]["amount_usd"].sum()
t1_cm    = cm[cm["country_role"] == "tier1"]["funding_usd"].sum()
print(f"\nTier1 total (clean): ${t1_clean/1e6:.1f}M")
print(f"Tier1 total (cm):    ${t1_cm/1e6:.1f}M")
assert round(t1_clean) == round(t1_cm), "MISMATCH: totals don't agree"
print("Totals match ✓")

print("\nSample clean rows (BHS):")
print(clean[clean["iso3"]=="BHS"].head(8)[
    ["iso3","year","month","source_country","source_org","dest_org","sector","amount_usd"]
].to_string(index=False))

Rows before dedup: 7054
Rows after dedup:  5610  (1444 exact duplicates removed)

fts_flows_clean.csv: (5610, 11)
source_country nulls remaining: 0

Top source_country values:
source_country
Multilateral     1510
United States    1326
Canada            432
Switzerland       359
Norway            286
Private           269
Sweden            195
Spain             186
Germany           114
France            107
Colombia           96
Japan              76

fts_flows_country_month.csv: (742, 9)

Tier1 total (clean): $2716.0M
Tier1 total (cm):    $2716.0M
Totals match ✓

Sample clean rows (BHS):
iso3  year  month source_country                      source_org                                 dest_org                            sector  amount_usd
 BHS  2019      3   Multilateral   United States Fund for UNICEF           United Nations Children's Fund                         Education       19800
 BHS  2019      9   Multilateral   United States Fund for UNICEF           United Nations Children's